# Constitutional AI and Self-Improvement Notebook

> Hands-on Build It and Exercises.

## Build It

The code implements three things in pure Python + numpy. A Constitutional AI self-critique loop. A rule-based reward checker for simple arithmetic. A minimal GRPO trainer that runs on a tiny language model from Lesson 04.

### Step 1: The Constitution

A list of principles. In production, each line would be richer and category-tagged. For the lesson, keep it short.

In [ ]:
```python

CONSTITUTION = [

    "The response must directly answer the question asked, without hedging.",

    "The response must not include unnecessary filler or padding.",

    "If the question has a single numeric answer, state the number plainly.",

    "The response must not refuse a reasonable, benign request.",

]

In [ ]:
```

### Step 2: Self-Critique and Revise

In a real system the model itself critiques. In the lesson we simulate a critic with a handwritten rubric so the pipeline runs without an LLM call.

In [ ]:
```python

def critique(response: str, principle: str) -> dict:

    problems = []

    if len(response.split()) > 40 and "plainly" in principle:

        problems.append("answer buried in extra prose")

    if response.strip().lower().startswith(("i can't", "i cannot", "as an ai")):

        problems.append("unwarranted refusal")

    if response.count(",") > 4:

        problems.append("too much hedging")

    return {"principle": principle, "problems": problems}

def revise(response: str, critique_result: dict) -> str:

    if "answer buried" in " ".join(critique_result["problems"]):

        return response.split(".")[-2].strip() + "."

    if "unwarranted refusal" in " ".join(critique_result["problems"]):

        return "Here is the answer: " + response.split(":")[-1].strip()

    return response

In [ ]:
```

The revise function is a stand-in. With a real LLM it would be a second prompt: "Given the critique, rewrite the response."

### Step 3: Rule-Based Rewards

For verifiable tasks, replace the critic entirely. This checker grades arithmetic answers.

In [ ]:
```python

import re

def reward_math(prompt: str, response: str) -> float:

    try:

        expected = eval(prompt.replace("What is ", "").replace("?", "").strip())

    except Exception:

        return 0.0

    numbers = re.findall(r"-?\d+", response)

    if not numbers:

        return 0.0

    return 1.0 if int(numbers[-1]) == expected else 0.0

def reward_format(response: str) -> float:

    return 1.0 if re.search(r"<answer>.*</answer>", response) else 0.0

In [ ]:
```

Two deterministic rules. No training data. No human labels. The combined reward is `reward_math + 0.1 * reward_format`, penalizing missing format without drowning out correctness.

### Step 4: Group-Relative Advantage

Given a list of rewards for a group of responses to the same prompt, compute the z-score:

In [ ]:
```python

import numpy as np

def group_relative_advantage(rewards: list[float]) -> np.ndarray:

    r = np.array(rewards, dtype=float)

    if r.std() < 1e-8:

        return np.zeros_like(r)

    return (r - r.mean()) / (r.std() + 1e-8)

In [ ]:
```

If every sample in the group has the same reward, the advantage is zero and no gradient signal flows. This is a feature. It tells you the prompt is either trivially solved or impossibly hard for the current policy, and the step should skip it.

### Step 5: GRPO Update

One step, symbolic gradient. In production this would be a torch autograd pass. Here we show the update rule directly.

In [ ]:
```python

def grpo_step(policy_logprobs: np.ndarray, ref_logprobs: np.ndarray,

              advantages: np.ndarray, beta: float = 0.01, clip_eps: float = 0.2) -> dict:

    ratios = np.exp(policy_logprobs - ref_logprobs)

    unclipped = ratios * advantages

    clipped = np.clip(ratios, 1 - clip_eps, 1 + clip_eps) * advantages

    policy_loss = -np.minimum(unclipped, clipped).mean()

    kl = (ref_logprobs - policy_logprobs).mean()

    total_loss = policy_loss + beta * kl

    return {

        "policy_loss": float(policy_loss),

        "kl": float(kl),

        "total_loss": float(total_loss),

        "mean_ratio": float(ratios.mean()),

    }

In [ ]:
```

This is PPO's clipped surrogate with one change: the advantages came from group-relative z-scores, not from a value function. No V(s) to train. No GAE. The group is the baseline.

### Step 6: Self-Improvement Round

Tie the pieces together. Sample a group, score each response with the rule, compute advantages, report the metrics you would feed into a real optimizer.

In [ ]:
```python

def self_improvement_round(prompts: list[str], policy_sampler, group_size: int = 8) -> dict:

    metrics = []

    for prompt in prompts:

        responses = [policy_sampler(prompt) for _ in range(group_size)]

        rewards = [reward_math(prompt, r) + 0.1 * reward_format(r) for r in responses]

        advantages = group_relative_advantage(rewards)

        best = responses[int(np.argmax(rewards))]

        metrics.append({

            "prompt": prompt,

            "mean_reward": float(np.mean(rewards)),

            "best_reward": float(np.max(rewards)),

            "std_reward": float(np.std(rewards)),

            "best_response": best,

            "advantages": advantages.tolist(),

        })

    return {"per_prompt": metrics,

            "overall_mean": float(np.mean([m["mean_reward"] for m in metrics]))}

In [ ]:
```

## Exercises

In [ ]:
1. Replace the handwritten critic in Step 2 with an LLM call. Use any local chat model. Measure how often the critique and revision actually improve the response versus leaving it unchanged.

2. Add a third constitutional principle about factuality. Run the pipeline on prompts that require factual claims (capitals, dates) and measure how many revisions remove factual errors versus introduce new ones.

3. Implement DPO on the preference pairs produced by CAI stage 2. Take 20 prompts, generate two responses each, have the critic pick a winner per pair, then run the DPO loss from Lesson 08. Compare to the GRPO path on the same data.

4. Add entropy regularization to the GRPO objective. The term `-alpha * entropy(policy)` with alpha=0.01 encourages diverse sampling. Measure whether it delays mode collapse across 5 rounds of self-improvement.

5. Build a process reward scorer for a two-step arithmetic problem. Given "What is (3+4)*5?", the model must show the intermediate 3+4=7 step. Grade the intermediate step separately from the final answer and compare PRM-weighted GRPO to pure ORM-weighted GRPO over 10 rounds.